In [1]:
import os

os.chdir(os.getenv("HOME_DIR"))

import random

import onnxruntime as ort
import torch
import yaml
from dotenv import load_dotenv
from ultralytics import YOLO

from src.utils.metrics import fps

load_dotenv()

True

In [2]:
handle_model_yaml = "yolo8_baseline.yaml"
dataset_yaml = "bdd100k.yaml"
yaml_path = os.path.join(
    os.getenv("HOME_DIR"),
    "config",
    "models",
    handle_model_yaml,
)
with open(yaml_path, "r") as file:
    args = yaml.safe_load(file)

In [3]:
DATA_DIR = os.path.join(
    os.getenv("HOME_DIR"), "config", "datasets", dataset_yaml
)  # Default dataset_name.yaml or personal_dataset_name.yaml
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))
PROJECT_DIR = os.path.join(
    os.getenv("HOME_DIR"), "results", "models", args["project_results_name"]
)
TESTING_IMG_DIR = os.path.join(
    os.getenv("HOME_DIR"), "data", "processed", "images", "test"
)  # Testing images directory

# **Load models**

In [4]:
onnx_path = os.path.join(PROJECT_DIR, "optimized", "best_optimized.onnx")
best_model_path = os.path.join(
    PROJECT_DIR,
    "optimized",
    "best_optimized.pt",
)
if not os.path.exists(best_model_path) and not os.path.exists(onnx_path):
    onnx_path = os.path.join(PROJECT_DIR, "train", "weights", "best.onnx")
    best_model_path = os.path.join(
        PROJECT_DIR,
        "train",
        "weights",
        "best.pt",
    )

# **Measure model size**

In [5]:
model_size = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"Model size: {model_size:.2f} MB")

Model size: 11.62 MB


# **Load ONNX model on CPU**

In [6]:
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
image_path = os.path.join(
    TESTING_IMG_DIR,
    os.listdir(TESTING_IMG_DIR)[random.randint(0, len(os.listdir(TESTING_IMG_DIR)))],
)
result = fps(sess, image_path)
print(f"FPS on CPU (edge simulation): {round(result['fps'], 3)}")
print(f"Min time inference {round(min(result['times']) * 1000, 3)} ms")
print(
    f"Mean time inference {round(sum(result['times']) / len(result['times']) * 1000, 3)} ms"
)
print(f"Max time inference {round(max(result['times']) * 1000, 3)} ms")

FPS on CPU (edge simulation): 61.283
Min time inference 16.004 ms
Mean time inference 16.318 ms
Max time inference 23.005 ms


# **Get profile on CPU time**

In [7]:
best_model = YOLO(best_model_path, task="detect", verbose=True).to(torch.device("cpu"))
real_input_torch = torch.from_numpy(
    result["real_input"].transpose((0, 2, 3, 1))
).permute(0, 3, 1, 2)

with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
    best_model(real_input_torch)
print(prof.key_averages().table(sort_by="self_cpu_time_total"))


0: 480x480 11 cars, 31.3ms
Speed: 0.0ms preprocess, 31.3ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 480)
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   aten::mkldnn_convolution        46.03%      22.027ms        46.85%      22.417ms     373.624us            60  
                             aten::uniform_        19.36%       9.265ms        19.36%       9.265ms      81.270us           114  
                                   aten::mm        10.86%       5.197ms        10.89%       5.210ms      45.703us           114  
                                aten::silu_         3.98%       1.906ms         3.98%       1.9